In [1]:
# ============================================================
# STEP 0 - PROCESSING FOR INDIAN VILLAGE 51
# Build graph, features, demographic ARD and membership matrix
# ============================================================

import os
import numpy as np
import pandas as pd
import networkx as nx

In [2]:
# ============================================================
# CELL 1 - Paths and parameters
# ============================================================

adj_path = "vil_adj.csv"
dem_path = "vil_dem.csv"

dataset_prefix = "village51"

min_circle_percent = 5
max_unique_for_categorical = 50

id_or_text_cols_to_drop = [
    "hhid",
    "resp_id",
    "native_name",
    "native_taluk",
    "native_district",
    "movecontact_name"
]

# Candidate variables to inspect as possible circles
candidate_circle_source_cols = [
    "caste",
    "mothertongue",
    "resp_gend"
]

# After inspecting distributions, choose final circle variables here.
# My first recommendation: start with caste only.
selected_circle_source_cols = [
    "caste",
    # "mothertongue",
    "resp_gend"
]

In [3]:
# ============================================================
# CELL 2 - Load adjacency matrix and build graph
# ============================================================

def load_village_graph_from_adjacency(adj_path):
    adj_df = pd.read_csv(adj_path, index_col=0)

    node_ids = [
        int(str(col).replace("V", ""))
        for col in adj_df.columns
    ]

    adj = adj_df.to_numpy()

    np.fill_diagonal(adj, 0)

    adj = ((adj + adj.T) > 0).astype(int)

    G = nx.from_numpy_array(adj)

    mapping = {
        i: node_ids[i]
        for i in range(len(node_ids))
    }

    G = nx.relabel_nodes(G, mapping)

    G = nx.Graph(G)
    G.add_nodes_from(node_ids)

    return G, node_ids, adj


G, node_ids, adj = load_village_graph_from_adjacency(adj_path)

print("=== Graph Info ===")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Self-loops:", nx.number_of_selfloops(G))
print("First nodes:", list(G.nodes())[:10])

=== Graph Info ===
Nodes: 309
Edges: 2172
Self-loops: 0
First nodes: [1, 2, 6, 11, 12, 19, 20, 43, 44, 45]


In [5]:
# ============================================================
# GRAPH METRICS
# ============================================================

degrees = np.array([degree for _, degree in G.degree()])

graph_metrics = {
    "Number of nodes": G.number_of_nodes(),
    "Number of edges": G.number_of_edges(),
    "Minimum degree": degrees.min(),
    "Maximum degree": degrees.max(),
    "Average degree": degrees.mean(),
    "Density": nx.density(G),
    "Average clustering coefficient": nx.average_clustering(G),
}

graph_metrics_df = pd.DataFrame(
    graph_metrics.items(),
    columns=["Metric", "Value"]
)

display(graph_metrics_df)

,Metric,Value
0,Number of nodes,309.000000
1,Number of edges,2172.000000
2,Minimum degree,3.000000
3,Maximum degree,43.000000
4,Average degree,14.058252
5,Density,0.045644
6,Average clustering coefficient,0.319869


In [88]:
# ============================================================
# CELL 3 - Save edge list
# ============================================================

def save_edges_from_graph(G, output_path):
    edges_df = pd.DataFrame(
        list(G.edges()),
        columns=["from", "to"]
    )

    edges_df.to_csv(
        output_path,
        sep=" ",
        header=False,
        index=False
    )

    print("Saved:", output_path)
    print("Edges:", len(edges_df))


edges_output_path = f"{dataset_prefix}.edges"

save_edges_from_graph(
    G,
    edges_output_path
)

Saved: village51.edges
Edges: 2172


In [89]:
# ============================================================
# CELL 4 - Load demographics and align with graph nodes
# ============================================================

def load_village_demographics(dem_path, node_ids):
    dem_df = pd.read_csv(dem_path)

    if dem_df.shape[0] != len(node_ids):
        raise ValueError(
            f"Demographics rows ({dem_df.shape[0]}) do not match "
            f"number of graph nodes ({len(node_ids)})."
        )

    dem_df = dem_df.copy()
    dem_df.index = node_ids
    dem_df.index.name = "node_id"

    return dem_df


dem_df = load_village_demographics(
    dem_path=dem_path,
    node_ids=node_ids
)

print("Demographics shape:", dem_df.shape)
display(dem_df.head())

Demographics shape: (309, 34)


,hhid,resp_id,resp_gend,resp_status,age,religion,caste,subcaste,mothertongue,speakother,...,movecontact_name,workflag,work_freq_type,occupation,privategovt,shgparticipate,savings,electioncard,rationcard,rationcard_colour
node_id,,,,,,,,,,,,,,,,,,,,,
1,51001.0,1.0,1.0,Head of Household,34.0,HINDUISM,GENERAL,LINGAYATH,KANNADA,Yes,...,NaN,Yes,DAYS PER WEEK,TEACHER,GOVERNMENT,No,Yes,Yes,Yes,RED
2,51001.0,2.0,2.0,Spouse of Head of Household,26.0,HINDUISM,GENERAL,LINGAYATH,KANNADA,No,...,NaN,No,NaN,NaN,NaN,No,Yes,Yes,Yes,RED
6,51001.0,6.0,1.0,Other,31.0,HINDUISM,GENERAL,LINGAYATH,KANNADA,No,...,NaN,Yes,DAYS PER WEEK,AGRICULTURIST,OWN LAND,No,Yes,Yes,Yes,RED
11,51002.0,1.0,1.0,Head of Household,48.0,HINDUISM,OBC,VOKKALIGA,KANNADA,No,...,NaN,Yes,DAYS PER WEEK,PETTY SHOP,PRIVATE BUSINESS,No,No,Yes,Yes,YELLOW
12,51002.0,2.0,2.0,Spouse of Head of Household,35.0,HINDUISM,OBC,VOKKALIGA,KANNADA,No,...,NaN,Yes,DAYS PER WEEK,AGRICULTURIST,OWN LAND,No,No,Yes,Yes,YELLOW


In [90]:
# ============================================================
# CELL 5 - Inspect candidate circle distributions
# ============================================================

def inspect_candidate_circle_distributions(dem_df, candidate_cols):
    summaries = {}

    for col in candidate_cols:
        if col not in dem_df.columns:
            print(f"\nWARNING: {col} not found in demographics.")
            continue

        counts = (
            dem_df[col]
            .fillna("Missing")
            .astype(str)
            .value_counts()
            .rename("num_nodes")
            .to_frame()
        )

        counts["percent_nodes"] = (
            counts["num_nodes"] / len(dem_df) * 100
        ).round(2)

        summaries[col] = counts

        print("\n" + "=" * 70)
        print(f"Distribution for candidate circle variable: {col}")
        print("=" * 70)
        display(counts)

    return summaries


candidate_circle_distributions = inspect_candidate_circle_distributions(
    dem_df=dem_df,
    candidate_cols=candidate_circle_source_cols
)


Distribution for candidate circle variable: caste


,num_nodes,percent_nodes
caste,,
OBC,115,37.22
SCHEDULED CASTE,82,26.54
GENERAL,81,26.21
SCHEDULED TRIBE,31,10.03



Distribution for candidate circle variable: mothertongue


,num_nodes,percent_nodes
mothertongue,,
KANNADA,299,96.76
TELUGU,10,3.24



Distribution for candidate circle variable: resp_gend


,num_nodes,percent_nodes
resp_gend,,
2.0,168,54.37
1.0,141,45.63


In [91]:
# ============================================================
# CELL 6 - Build features from demographics
# Exclude selected circle variables from DGI features
# Everything categorical; age discretized
# ============================================================

def build_features_from_demographics(
    dem_df,
    drop_cols=None,
    circle_cols_to_exclude=None,
    max_unique_for_categorical=50
):
    if drop_cols is None:
        drop_cols = []

    if circle_cols_to_exclude is None:
        circle_cols_to_exclude = []

    df = dem_df.copy()

    cols_to_drop = list(set(drop_cols + circle_cols_to_exclude))

    existing_drop_cols = [
        col for col in cols_to_drop
        if col in df.columns
    ]

    if len(existing_drop_cols) > 0:
        print("Dropping columns from DGI features:", existing_drop_cols)

    df = df.drop(columns=existing_drop_cols)

    nunique = df.nunique(dropna=True)
    constant_cols = nunique[nunique <= 1].index.tolist()

    if len(constant_cols) > 0:
        print("Dropping constant columns:", constant_cols)

    df = df.drop(columns=constant_cols)

    processed_df = pd.DataFrame(index=df.index)

    for col in df.columns:

        if col == "age":
            numeric_age = pd.to_numeric(
                df[col],
                errors="coerce"
            )

            median_age = numeric_age.median()

            if pd.isna(median_age):
                median_age = 0

            processed_df[col] = numeric_age.fillna(median_age)

        else:
            processed_df[col] = (
                df[col]
                .fillna("Missing")
                .astype(str)
            )

    df = processed_df

    if "age" in df.columns:
        df["age_group"] = pd.cut(
            df["age"],
            bins=[-np.inf, 30, 45, 60, np.inf],
            labels=[
                "age_30_or_less",
                "age_31_45",
                "age_46_60",
                "age_61_plus"
            ]
        ).astype(str)

        df = df.drop(columns=["age"])

        print("Age discretized into age_group and continuous age removed.")

    categorical_cols = []

    for col in df.columns:
        if df[col].nunique(dropna=True) <= max_unique_for_categorical:
            categorical_cols.append(col)

    dropped_high_cardinality = [
        col for col in df.columns
        if col not in categorical_cols
    ]

    if len(dropped_high_cardinality) > 0:
        print(
            "Dropping high-cardinality categorical columns:",
            dropped_high_cardinality
        )

    features_df = pd.get_dummies(
        df[categorical_cols],
        columns=categorical_cols,
        drop_first=False,
        dtype=int
    )

    features_df.index = dem_df.index
    features_df.index.name = "node_id"

    return features_df


features_aligned = build_features_from_demographics(
    dem_df=dem_df,
    drop_cols=id_or_text_cols_to_drop,
    circle_cols_to_exclude=selected_circle_source_cols,
    max_unique_for_categorical=max_unique_for_categorical
)

features_interpretation = features_aligned.copy()

print("DGI features shape:", features_aligned.shape)
display(features_aligned.head())

features_output_path = f"features_aligned_{dataset_prefix}.csv"
features_interpretation_output_path = f"features_interpretation_{dataset_prefix}.csv"

features_aligned.to_csv(features_output_path)
features_interpretation.to_csv(features_interpretation_output_path)

print("Saved:", features_output_path)
print("Saved:", features_interpretation_output_path)

Dropping columns from DGI features: ['hhid', 'caste', 'resp_id', 'resp_gend', 'movecontact_name', 'native_district', 'native_taluk', 'native_name']
Dropping constant columns: ['religion', 'urdu', 'otherlang', 'work_freq_type']
Age discretized into age_group and continuous age removed.
DGI features shape: (309, 136)


,resp_status_Head of Household,resp_status_Other,resp_status_Spouse of Head of Household,subcaste_ADI DRAVIDA,subcaste_ADI KARNATAKA,subcaste_BAJENTHRI,subcaste_BHOVI,subcaste_EDIGA,subcaste_EDIGARU,subcaste_GOLLA,...,rationcard_colour_APL,rationcard_colour_BLUE,rationcard_colour_GREEN,rationcard_colour_Missing,rationcard_colour_RED,rationcard_colour_YELLOW,age_group_age_30_or_less,age_group_age_31_45,age_group_age_46_60,age_group_age_61_plus
node_id,,,,,,,,,,,,,,,,,,,,,
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
2,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,1,0,1,0,0,0
6,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
11,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,1,0
12,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,1,0,0


Saved: features_aligned_village51.csv
Saved: features_interpretation_village51.csv


In [92]:
# ============================================================
# CELL 7 - Build circle membership from selected variables
# ============================================================

def build_circle_membership_from_selected_variables(
    dem_df,
    selected_circle_source_cols,
    min_percent=5
):
    circle_membership_parts = []

    for col in selected_circle_source_cols:
        if col not in dem_df.columns:
            raise ValueError(f"Selected circle variable not found: {col}")

        series = (
            dem_df[col]
            .fillna("Missing")
            .astype(str)
        )

        dummies = pd.get_dummies(
            series,
            prefix=col,
            drop_first=False,
            dtype=int
        )

        dummies.index = dem_df.index
        circle_membership_parts.append(dummies)

    membership_matrix = pd.concat(
        circle_membership_parts,
        axis=1
    )

    circle_summary_df = pd.DataFrame({
        "num_nodes": membership_matrix.sum(axis=0)
    })

    circle_summary_df["percent_nodes"] = (
        circle_summary_df["num_nodes"] / membership_matrix.shape[0] * 100
    ).round(2)

    circle_summary_df = circle_summary_df.sort_values(
        "num_nodes",
        ascending=False
    )

    selected_circles = circle_summary_df[
        circle_summary_df["percent_nodes"] >= min_percent
    ].index.tolist()

    filtered_membership_matrix = membership_matrix[selected_circles].copy()

    filtered_summary_df = circle_summary_df.loc[selected_circles].copy()

    print("\n=== Circle variables selected ===")
    print(selected_circle_source_cols)

    print("\n=== Raw circle summary ===")
    display(circle_summary_df)

    print("\n=== Filtered circle summary ===")
    print(f"Minimum size: {min_percent}% of nodes")
    display(filtered_summary_df)

    print("Membership matrix shape:", filtered_membership_matrix.shape)

    return filtered_membership_matrix, circle_summary_df, filtered_summary_df


membership_matrix, circle_summary_df, filtered_circle_summary_df = (
    build_circle_membership_from_selected_variables(
        dem_df=dem_df,
        selected_circle_source_cols=selected_circle_source_cols,
        min_percent=min_circle_percent
    )
)

display(membership_matrix.head())


=== Circle variables selected ===
['caste', 'resp_gend']

=== Raw circle summary ===


,num_nodes,percent_nodes
resp_gend_2.0,168,54.37
resp_gend_1.0,141,45.63
caste_OBC,115,37.22
caste_SCHEDULED CASTE,82,26.54
caste_GENERAL,81,26.21
caste_SCHEDULED TRIBE,31,10.03



=== Filtered circle summary ===
Minimum size: 5% of nodes


,num_nodes,percent_nodes
resp_gend_2.0,168,54.37
resp_gend_1.0,141,45.63
caste_OBC,115,37.22
caste_SCHEDULED CASTE,82,26.54
caste_GENERAL,81,26.21
caste_SCHEDULED TRIBE,31,10.03


Membership matrix shape: (309, 6)


,resp_gend_2.0,resp_gend_1.0,caste_OBC,caste_SCHEDULED CASTE,caste_GENERAL,caste_SCHEDULED TRIBE
node_id,,,,,,
1,0,1,0,0,1,0
2,1,0,0,0,1,0
6,0,1,0,0,1,0
11,0,1,1,0,0,0
12,1,0,1,0,0,0


In [93]:
# ============================================================
# CELL 8 - Compute ARD from selected circle membership
# ============================================================

def compute_ard_matrix_from_membership(G, membership_matrix):
    nodes = list(G.nodes())

    membership_matrix = membership_matrix.loc[nodes]

    ard_matrix = pd.DataFrame(
        0,
        index=nodes,
        columns=membership_matrix.columns
    )

    for node in nodes:
        neighbors = list(G.neighbors(node))

        if len(neighbors) > 0:
            ard_matrix.loc[node] = membership_matrix.loc[neighbors].sum(axis=0)

    ard_matrix = ard_matrix.sort_index()
    ard_matrix.index.name = "node_id"

    return ard_matrix


ard_matrix = compute_ard_matrix_from_membership(
    G=G,
    membership_matrix=membership_matrix
)

print("\n=== ARD Matrix Info ===")
print("ARD shape:", ard_matrix.shape)
print("First columns:", list(ard_matrix.columns[:10]))
display(ard_matrix.head())


=== ARD Matrix Info ===
ARD shape: (309, 6)
First columns: ['resp_gend_2.0', 'resp_gend_1.0', 'caste_OBC', 'caste_SCHEDULED CASTE', 'caste_GENERAL', 'caste_SCHEDULED TRIBE']


,resp_gend_2.0,resp_gend_1.0,caste_OBC,caste_SCHEDULED CASTE,caste_GENERAL,caste_SCHEDULED TRIBE
node_id,,,,,,
1,1,18,4,1,14,0
2,12,5,2,2,12,1
6,1,11,5,0,7,0
11,2,24,11,4,10,1
12,12,2,9,0,4,1


In [94]:
# ============================================================
# CELL 9 - Validate ARD
# ============================================================

def validate_ard_from_membership(G, ard_matrix, membership_matrix):
    nodes = list(G.nodes())

    membership_matrix = membership_matrix.loc[nodes]

    expected_sums = pd.Series(0, index=nodes, dtype=int)

    for node in nodes:
        neighbors = list(G.neighbors(node))

        if len(neighbors) > 0:
            expected_sums.loc[node] = membership_matrix.loc[neighbors].sum().sum()

    ard_sums = ard_matrix.sum(axis=1)

    expected_sums = expected_sums.sort_index()
    ard_sums = ard_sums.sort_index()

    matches = ard_sums == expected_sums

    print("¿ARD correcta using selected circles?:", matches.all())

    if not matches.all():
        print("Number of mismatches:", (~matches).sum())

    return matches


validate_ard_from_membership(
    G=G,
    ard_matrix=ard_matrix,
    membership_matrix=membership_matrix
)

¿ARD correcta using selected circles?: True


node_id
1       True
2       True
6       True
11      True
12      True
        ... 
1027    True
1029    True
1030    True
1035    True
1036    True
Length: 309, dtype: bool

In [95]:
# ============================================================
# CELL 10 - Compute total_prop from selected circle membership
# ============================================================

def compute_total_prop_from_membership(G, membership_matrix):
    edges = list(G.edges())

    if len(edges) == 0:
        return 0

    membership_matrix = membership_matrix.loc[list(G.nodes())]

    feature_ties = np.zeros(membership_matrix.shape[1])

    for k, circle_name in enumerate(membership_matrix.columns):
        feature_ties[k] = sum(
            (membership_matrix.loc[u, circle_name] == 1) or
            (membership_matrix.loc[v, circle_name] == 1)
            for u, v in edges
        )

    total_prop = feature_ties.sum() / len(edges)

    return total_prop


total_prop = compute_total_prop_from_membership(
    G=G,
    membership_matrix=membership_matrix
)

print("Total prop:", total_prop)

Total prop: 2.625230202578269


In [96]:
# ============================================================
# CELL 11 - Save processed files
# ============================================================

ard_matrix_file_path = f"ard_matrix_{dataset_prefix}.csv"
membership_matrix_file_path = f"membership_matrix_{dataset_prefix}.csv"
total_prop_file_path = f"total_prop_{dataset_prefix}.txt"
circle_summary_file_path = f"circle_summary_{dataset_prefix}.csv"
filtered_circle_summary_file_path = f"filtered_circle_summary_{dataset_prefix}.csv"

ard_matrix.to_csv(ard_matrix_file_path)
membership_matrix.to_csv(membership_matrix_file_path)
circle_summary_df.to_csv(circle_summary_file_path)
filtered_circle_summary_df.to_csv(filtered_circle_summary_file_path)

with open(total_prop_file_path, "w") as f:
    f.write(f"Total Proportion: {total_prop}\n")

print("Saved files:")
print(edges_output_path)
print(features_output_path)
print(features_interpretation_output_path)
print(ard_matrix_file_path)
print(membership_matrix_file_path)
print(total_prop_file_path)
print(circle_summary_file_path)
print(filtered_circle_summary_file_path)

Saved files:
village51.edges
features_aligned_village51.csv
features_interpretation_village51.csv
ard_matrix_village51.csv
membership_matrix_village51.csv
total_prop_village51.txt
circle_summary_village51.csv
filtered_circle_summary_village51.csv
